# Solar Energy Data Cleaning : Olivia Pozytko


In [5]:

from pathlib import Path
import pandas as pd
import geopandas as gpd

# STEP 1: Set file paths
data_file = Path("data/solar_raw.csv")
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

# STEP 2: Import solar data CSV
solar = pd.read_csv(data_file)

print("Original number of facilities:", len(solar))
print("Original columns:", solar.columns.tolist())

# STEP 3: Keep the important columns
columns = [
    "case_id",
    "p_name",
    "p_state",
    "p_county",
    "ylat",
    "xlong",
    "p_cap_ac", #Electrical Capacity \\ best to use to compare w/ transmission
    "p_cap_dc", #solar output 
    "p_year"
]

solar = solar[columns].copy()

# STEP 4: Rename columns
solar = solar.rename(columns={
    "case_id": "facility_id",
    "p_name": "facility_name",
    "p_state": "state",
    "p_county": "county",
    "ylat": "latitude",
    "xlong": "longitude",
    "p_cap_ac": "capacity_mw_ac",
    "p_cap_dc": "capacity_mw_dc",
    "p_year": "year"
})

# STEP 5: Remove duplicate facilities
solar = solar.drop_duplicates(subset="facility_id")

# STEP 6: Convert numeric columns
numeric_cols = [
    "latitude",
    "longitude",
    "capacity_mw_ac",
    "capacity_mw_dc",
    "year"
]

for col in numeric_cols:
    solar[col] = pd.to_numeric(
        solar[col], errors="coerce"
    )

# STEP 7: Remove missing or invalid coordinates
solar = solar.dropna(
    subset=["latitude", "longitude"]
)

solar = solar[
    solar["latitude"].between(-90, 90) &
    solar["longitude"].between(-180, 180)
].copy()

# STEP 8: Filter to North Carolina
solar["state"] = solar["state"].str.strip().str.upper()
solar = solar[solar["state"] == "NC"].copy()

# STEP 9: Add energy source label
solar["energy_type"] = "solar"

# STEP 10: Create geographic points
solar_geo = gpd.GeoDataFrame(
    solar,
    geometry=gpd.points_from_xy(
        solar["longitude"],
        solar["latitude"]
    ),
    crs="EPSG:4326"
)

# STEP 11: Save cleaned files
solar.to_csv(
    output_folder / "solar_nc_clean.csv",
    index=False
)

solar_geo.to_file(
    output_folder / "solar_nc.geojson",
    driver="GeoJSON"
)

print("Cleaned NC solar facilities:", len(solar))
print("Files saved successfully!")


Original number of facilities: 6611
Original columns: ['case_id', 'multi_poly', 'eia_id', 'p_state', 'p_county', 'ylat', 'xlong', 'p_area', 'p_img_date', 'p_dig_conf', 'p_name', 'p_year', 'p_pwr_reg', 'p_tech_pri', 'p_tech_sec', 'p_sys_type', 'p_axis', 'p_azimuth', 'p_tilt', 'p_battery', 'p_cap_ac', 'p_cap_dc', 'p_type', 'p_agrivolt', 'p_comm', 'p_zscore']
Cleaned NC solar facilities: 760
Files saved successfully!


Inspect the CSV is correctly cleaned

In [4]:
solar.head()

,facility_id,facility_name,state,county,latitude,longitude,capacity_mw_ac,capacity_mw_dc,year,energy_type
263,407366,Davidson III,NC,Davidson,35.7420,-80.2924,4.5,5.0,2010,solar
311,406473,Oak Solar,NC,Northampton,36.5361,-77.7661,120.0,201.0,2023,solar
314,400471,Haynes Farm,NC,Lincoln,35.5260,-81.2351,5.0,6.0,2013,solar
315,400725,Sonne One,NC,Wayne,35.4388,-77.8636,5.0,7.0,2016,solar
321,400464,Arndt Farm,NC,Catawba,35.7785,-81.1767,5.0,6.5,2012,solar


In [6]:
print(solar.shape)

(760, 10)


In [ ]:
#Check for NA values
print(solar.isnull().sum())

facility_id       0
facility_name     0
state             0
county            0
latitude          0
longitude         0
capacity_mw_ac    0
capacity_mw_dc    0
year              0
energy_type       0
dtype: int64


In [ ]:
#Find duplicates
print(solar.duplicated().sum())

0


In [13]:
#NC Solar
nc_solar = solar[solar["state"] == "NC"].copy()

print(nc_solar.shape)

(760, 10)


In [14]:
print(solar.columns.tolist())

['facility_id', 'facility_name', 'state', 'county', 'latitude', 'longitude', 'capacity_mw_ac', 'capacity_mw_dc', 'year', 'energy_type']
